# Project 3: Advection-diffusion equations

Put a drop of ink in a river. The flow carries it downstream while it slowly spreads out, and if the ink
reacted chemically with the water it could also grow or fade away. These three effects, *advection*,
*diffusion* and *reaction*, are described by the one-dimensional advection-diffusion equation

\begin{equation}
  \frac{\partial u}{\partial t} + V \frac{\partial u}{\partial x} = D \frac{\partial^2 u}{\partial x^2} + f(u)
  \tag{1}
\end{equation}

for a field $u(x, t)$ defined on a domain $x \in [0, L]$. Here $V$ is the advection velocity, $D$ the diffusion
coefficient and $f(u)$ a source term (losses, a chemical reaction, a nonlinear saturation...). We are given the
initial condition $u(x, 0) = u_0(x)$ and boundary conditions at $x = 0$ and $x = L$, which depend on the physical
problem:

- periodic: $u(0, t) = u(L, t)$
- Neumann: $\partial_x u(0, t) = \alpha$ and $\partial_x u(L, t) = \beta$ (for $\alpha = \beta = 0$, nothing flows in or out)
- Dirichlet: $u(0, t) = \alpha$ and $u(L, t) = \beta$

Equation (1) shows up all over physics: the heat equation ($u$ is the temperature), the transport of a tracer in
a fluid ($u$ is a concentration), the propagation of a laser pulse in an optical fiber ($u$ is the envelope of the
electric field and $D$ is imaginary), the time-dependent Schrödinger equation ($u$ is the wave function and
$D = i\hbar/2m$), or the Ginzburg-Landau equation ($u$ is the amplitude of a pattern, e.g. convection rolls).

In this project we will learn two general methods to integrate such *parabolic* partial differential equations:

- **Part 1**: a spectral method, based on Fourier transforms, very accurate for linear problems with periodic boundary conditions,
- **Part 2**: the Crank-Nicolson finite-difference scheme, which handles any boundary condition and nonlinear terms,

and then use them to explore some beautiful physics: invading fronts (**Part 3**) and spatiotemporal chaos in the
complex Ginzburg-Landau equation (**Part 4**).

In [ ]:
# Common imports
%matplotlib inline
import time
from collections.abc import Callable

import numpy as np
import scipy.linalg as la
import scipy.sparse as sparse

import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.image import AxesImage
from IPython.display import HTML

PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#56B4E9", "#E69F00", "#F0E442"]
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (6.5, 4.2),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "axes.edgecolor": "#898781",
    "axes.linewidth": 0.9,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "axes.axisbelow": True,
    "animation.html": "jshtml",    # animations are rendered as interactive JavaScript widgets
    "animation.embed_limit": 60,   # in MB
})

# the random number generator
rng = np.random.default_rng(seed=2026)

# colormaps: sequential for u >= 0, diverging for signed fields, cyclic for phases
SEQ_CMAP = LinearSegmentedColormap.from_list("sequential", ["#f7f6f3", "#56B4E9", "#0072B2", "#0b2545"])
DIV_CMAP = LinearSegmentedColormap.from_list("diverging", ["#0072B2", "#f0efec", "#D55E00"])
PHASE_CMAP = "twilight"

# to draw a field u(x, t) as a space-time map
def show_spacetime(x: np.ndarray, t: np.ndarray, u: np.ndarray, ax: plt.Axes | None = None,
                   cmap=SEQ_CMAP, label: str | None = None, **kwargs) -> AxesImage:
    """Draw u[n, j] = u(x_j, t_n) as a map, with x horizontal and t increasing upwards."""
    ax = ax if ax is not None else plt.gca()
    image = ax.imshow(u, origin="lower", aspect="auto", cmap=cmap,
                      extent=(x[0], x[-1], t[0], t[-1]), **kwargs)
    ax.set(xlabel="$x$", ylabel="$t$")
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.figure.colorbar(image, ax=ax, label=label, pad=0.02)
    return image

## Part 1: The spectral method

Let us start with one of the simplest and most accurate ways to integrate a partial differential equation: a
*spectral method*, where the solution is decomposed into Fourier modes

\begin{equation}
  \hat u(k, t) = \int_{-\infty}^{+\infty} u(x, t)\, e^{-ikx}\, dx
  \qquad
  u(x, t) = \frac{1}{2\pi} \int_{-\infty}^{+\infty} \hat u(k, t)\, e^{ikx}\, dk
\end{equation}

The Fourier basis is naturally suited to periodic boundary conditions. Here we consider a pulse in an infinite
domain, and we simply choose a numerical box large enough that the periodic copies of the pulse do not see each
other. We also take $f(u) = 0$. The magic of Fourier modes is that $e^{ikx}$ is an eigenfunction of both
$\partial_x$ and $\partial_x^2$: in Fourier space, Eq. (1) becomes a set of independent ordinary differential
equations, one for each wavenumber $k$,

\begin{equation}
  \frac{\partial \hat u}{\partial t} = -\big(D k^2 + i V k\big)\, \hat u
  \qquad \Longrightarrow \qquad
  \hat u(k, t) = \hat u(k, 0)\, e^{-D k^2 t}\, e^{-i V k t}
  \tag{P}
\end{equation}

This solution has a nice physical reading. The factor $e^{-Dk^2t}$ damps every mode, and the short wavelengths
(large $k$) die first: diffusion smooths things out. The factor $e^{-iVkt}$ is a pure phase, it just translates
the profile by $Vt$: this is advection.

### Numerical implementation

On a computer, the Fourier transforms are replaced by their discrete versions, computed by the fast Fourier
transform (FFT) in only $O(N \log N)$ operations. We discretize time and space as

\begin{equation}
  t_n = n\, \Delta t, \qquad x_j = j\, \Delta x, \quad j = 0, \dots, N-1, \qquad \Delta x = \frac{L}{N}
\end{equation}

and write $U^n_j \simeq u(x_j, t_n)$ for the numerical solution. Note that the box is periodic, so the point
$x = L$ is the same as $x = 0$ and must not be part of the grid. The wavenumbers are

\begin{equation}
  k_m = m\, \Delta k, \quad m = -\frac{N}{2}, \dots, \frac{N}{2} - 1, \qquad \Delta k = \frac{2\pi}{N \Delta x}
\end{equation}

and the largest one is the Nyquist wavenumber $k_c = \pi / \Delta x$: a wave shorter than two grid spacings
cannot be represented on the grid.

*Note*: FFT routines store the wavenumbers in a peculiar order. For `A = np.fft.fft(a)`, the zero mode is `A[0]`,
the positive wavenumbers are in `A[1:N//2]` and the negative ones in `A[N//2:]`. The function
`np.fft.fftfreq(N, d=dx)` returns the frequencies $k / 2\pi$ in that same order.

*Note*: FFTs are fastest when $N$ is a product of small primes, ideally a power of 2.

### Algorithm

To go from $U^n$ to $U^{n+1}$:

1. Compute $\hat U^n$ by applying the FFT to $U^n$.
2. Multiply by the propagator: $\hat U^{n+1}_m = \hat U^n_m\, \exp\big(-(D k_m^2 + i V k_m)\, \Delta t\big)$.
3. Compute $U^{n+1}$ by applying the inverse FFT to $\hat U^{n+1}$.

For this linear problem we could jump from $t = 0$ to any time $t$ in a single step. Doing many small steps
becomes useful as soon as there is a source term $f(u)$, which can then be handled in real space between two
Fourier steps (this is the idea of *split-step* methods).

### 1.1 The initial condition

We will use a Gaussian pulse as initial condition. When a function has several constant parameters, it is
often convenient to use a class instead of a function: the constant parameters are given once to the
constructor `__init__` and stored in the object, while the variable $x$ is passed to the special method
`__call__`. This method makes the object *callable*: `gauss(x)` is the same as `gauss.__call__(x)`.

Write a class for the Gaussian $u_0(x) = U_0 \exp\big(-(x - x_0)^2 / \sigma^2\big)$:

```python
class Gaussian:
    """A Gaussian pulse of given width and amplitude, centered at x0."""

    def __init__(self, width: float, amplitude: float, x0: float) -> None:
        self.width = width
        ...

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return ...
```

You can then create a Gaussian and use it like a function:

```python
gauss = Gaussian(width=1, amplitude=1, x0=0.5)
gauss(0.5)                       # 1.0
gauss(np.array([0, 1, 2, 3]))    # array([0.77880078, 0.77880078, 0.10539922, 0.00193045])
```

Plot your Gaussian to check that it is correct.

### Solution

In [ ]:
class Gaussian:
    """A Gaussian pulse of given width and amplitude, centered at x0."""

    def __init__(self, width: float, amplitude: float, x0: float) -> None:
        self.width = width
        self.amplitude = amplitude
        self.x0 = x0

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.amplitude * np.exp(-((x - self.x0) / self.width) ** 2)


gauss = Gaussian(width=1, amplitude=1, x0=0.5)
print(gauss(0.5), gauss(np.array([0, 1, 2, 3])))

x = np.linspace(-3, 4, 400)
fig, ax = plt.subplots()
ax.plot(x, gauss(x), label=r"$U_0\, e^{-(x - x_0)^2 / \sigma^2}$")
ax.axvline(gauss.x0, color="#898781", lw=1)
ax.annotate("", xy=(gauss.x0 + gauss.width, 1 / np.e), xytext=(gauss.x0, 1 / np.e),
            arrowprops=dict(arrowstyle="<->", color=PALETTE[1], lw=1.5, shrinkA=0, shrinkB=0))
ax.text(gauss.x0 + gauss.width / 2, 1 / np.e + 0.03, r"$\sigma$", color=PALETTE[1], ha="center")
ax.text(gauss.x0 + 0.08, 0.03, "$x_0$", color="#52514e")
ax.set(xlabel="$x$", ylabel="$u_0(x)$", ylim=(0, 1.1), title="A Gaussian pulse")
ax.legend(loc="upper right");

The width $\sigma$ is the distance from the center at which the pulse has dropped to $U_0/e$. By the way, Python
can write the `__init__` for us when a class mostly stores data, with a *dataclass*:

```python
from dataclasses import dataclass

@dataclass
class Gaussian:
    width: float
    amplitude: float
    x0: float

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.amplitude * np.exp(-((x - self.x0) / self.width) ** 2)
```

### 1.2 A class to advance the solution from $t$ to $t + \Delta t$

This class should create the spatial grid $x_j$ and the wavenumbers $k_m$, store the parameters $D$ and $V$, and
provide a way to set the initial condition. Its method `step(dt)` evolves $u(x, t) \to u(x, t + \Delta t)$ with
the three steps of the algorithm above:

1. transform $u(x, t) \to \hat u(k, t)$ with the FFT,
2. advance by $\Delta t$ using Eq. (P),
3. transform $\hat u(k, t + \Delta t) \to u(x, t + \Delta t)$ with the inverse FFT.

```python
class Spectral:
    def __init__(self, L: float, n_points: int, D: float, V: float) -> None:
        ...

    def initialize(self, func: Callable[[np.ndarray], np.ndarray]) -> None:
        ...

    def step(self, dt: float) -> None:
        ...
```

*Hints*: `np.linspace(0, L, n_points, endpoint=False)` creates a periodic grid without the point $x = L$.
The inverse FFT returns a complex array even if $u$ is real: its imaginary part is only round-off noise and can
be dropped.

### Solution

In [ ]:
class Spectral:
    """Solve du/dt + V du/dx = D d^2u/dx^2 on the periodic box [0, L) with Fourier modes."""

    def __init__(self, L: float, n_points: int, D: float, V: float) -> None:
        self.L, self.D, self.V = L, D, V
        # x = L is the same point as x = 0, so it is not part of the grid
        self.x, self.dx = np.linspace(0, L, n_points, endpoint=False, retstep=True)
        self.k = 2 * np.pi * np.fft.fftfreq(n_points, d=self.dx)
        self.u = np.zeros(n_points)
        self.t = 0.0

    def initialize(self, func: Callable[[np.ndarray], np.ndarray]) -> None:
        """Set the initial condition u(x, 0) = func(x)."""
        self.u = func(self.x)
        self.t = 0.0

    def step(self, dt: float) -> None:
        """Advance the solution from t to t + dt."""
        u_hat = np.fft.fft(self.u)
        u_hat *= np.exp(-(self.D * self.k**2 + 1j * self.V * self.k) * dt)
        self.u = np.fft.ifft(u_hat).real  # the imaginary part is only round-off noise
        self.t += dt

Let's look at the grids on a tiny example with $L = 2\pi$ and $N = 8$, so that $\Delta k = 1$. Note the order
of the wavenumbers returned by `fftfreq`: first the positive ones, then the negative ones.

In [ ]:
demo = Spectral(L=2 * np.pi, n_points=8, D=0.0, V=0.0)
print("x / pi =", demo.x / np.pi)
print("k      =", demo.k)

### 1.3 Time evolution

Write a function `advance(spectral, T, n_steps)` that evolves the solution from $t$ to $t + T$ using
`n_steps` steps.

### Solution

In [ ]:
def advance(spectral: Spectral, T: float, n_steps: int) -> None:
    """Evolve the solution from t to t + T with n_steps steps of size T / n_steps."""
    dt = T / n_steps
    for _ in range(n_steps):
        spectral.step(dt)

### 1.4 Check the results

1. Choose a problem that you can solve exactly, for example the pure diffusion of a tracer ($V = 0$). Starting
   from the Gaussian $u_0(x)$ above, the exact solution is

   \begin{equation}
     u(x, t) = \frac{U_0}{\sqrt{1 + 4Dt/\sigma^2}} \exp\left(-\frac{(x - x_0)^2}{\sigma^2 + 4Dt}\right)
   \end{equation}

   The pulse gets wider and lower, but its area $\int u\, dx = \sqrt{\pi}\, U_0 \sigma$ stays the same. Plot
   the numerical solution at a few times and compare with the exact solution. How does the error depend on the
   number of steps? Why?

2. Plot $|\hat U(k)|$ as a function of $k$ at a few times, on a logarithmic scale. Which modes disappear first?

3. Check the purely advective case ($D = 0$). Any function of $x - Vt$ solves $u_t + V u_x = 0$, so the pulse
   should travel without changing its shape. What happens when it reaches the edge of the box?

4. Finally, combine advection and diffusion and draw $u(x, t)$ as a space-time map using the function
   `show_spacetime` defined at the top of the notebook.

### Solution

We write the exact solution as a subclass of `Gaussian`: it is a Gaussian whose width and amplitude change with
time, and at $t = 0$ it is the initial condition. In the box $L = 20$, the pulse stays far enough from the edges
for its periodic copies to be negligible.

In [ ]:
class DiffusingGaussian(Gaussian):
    """Exact solution of the diffusion equation on the infinite line, for a Gaussian initial condition."""

    def __init__(self, width: float, amplitude: float, x0: float, D: float) -> None:
        super().__init__(width, amplitude, x0)
        self.D = D

    def __call__(self, x: np.ndarray, t: float = 0.0) -> np.ndarray:
        width_t = np.sqrt(self.width**2 + 4 * self.D * t)
        return self.amplitude * self.width / width_t * np.exp(-((x - self.x0) / width_t) ** 2)


L, n_points, D = 20.0, 512, 0.1
exact = DiffusingGaussian(width=0.4, amplitude=1.0, x0=L / 2, D=D)
spectral = Spectral(L, n_points, D=D, V=0.0)
spectral.initialize(exact)

times = [0, 0.25, 1, 2.5, 5]
colors = SEQ_CMAP(np.linspace(0.35, 1, len(times)))
k_sorted = np.fft.fftshift(spectral.k)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4), layout="constrained")
for t, color in zip(times, colors):
    advance(spectral, t - spectral.t, n_steps=10)
    u_hat = np.fft.fftshift(np.abs(np.fft.fft(spectral.u))) * spectral.dx
    ax1.plot(spectral.x, spectral.u, color=color, label=f"$t = {t:g}$")
    ax1.plot(spectral.x, exact(spectral.x, t), "--", color="#0b0b0b", lw=1)
    ax2.semilogy(k_sorted, u_hat, color=color, label=f"$t = {t:g}$")
ax1.plot([], [], "--", color="#0b0b0b", lw=1, label="exact")
ax1.set(xlabel="$x$", ylabel="$u(x, t)$", xlim=(6, 14), title="Diffusion of a Gaussian pulse")
ax1.legend(loc="upper right", fontsize=10)
ax2.set(xlabel="wavenumber $k$", ylabel=r"$|\hat u(k, t)|$", ylim=(1e-18, 3), xlim=(-k_sorted.max(), k_sorted.max()),
        title="Fourier spectrum")
ax2.text(-0.95 * k_sorted.max(), 1e-14, "round-off level", color="#52514e", fontsize=10);

The numerical solution sits right on top of the exact one. In Fourier space, the spectrum of a Gaussian is a
Gaussian, $|\hat u(k, t)| \propto e^{-k^2 \sigma^2/4}\, e^{-D k^2 t}$, which becomes narrower and narrower:
diffusion acts as a low-pass filter that erases the large wavenumbers first. Once a mode falls below
$\sim 10^{-16}$, it is lost in the round-off noise of the double precision numbers.

How large is the error, and how does it depend on the number of steps?

In [ ]:
T = 5.0
for n_steps in [1, 10, 100, 1000]:
    spectral.initialize(exact)
    advance(spectral, T, n_steps)
    error = np.max(np.abs(spectral.u - exact(spectral.x, T)))
    area = np.sum(spectral.u) * spectral.dx
    print(f"{n_steps:5d} steps:  max error = {error:.1e},  area = {area:.12f}  (exact: {np.sqrt(np.pi) * 0.4:.12f})")

The error is at the level of the machine precision whatever the number of steps: Eq. (P) is the *exact* solution
of the discretized problem, so the time step introduces no error at all. Only the round-off errors accumulate
slowly with the number of FFTs. The area is conserved because the mode $k = 0$, which is the integral of $u$, is
never damped. Such a perfect accuracy is the strength of spectral methods for linear problems with constant
coefficients.

Now the pure advection. To compare with the exact solution when the pulse crosses the edge of the periodic box, we
measure the distance to the center of the pulse modulo $L$.

In [ ]:
L, n_points, V = 10.0, 256, 1.0
pulse = Gaussian(width=0.4, amplitude=1.0, x0=2.5)

def advected_pulse(x: np.ndarray, t: float) -> np.ndarray:
    """The pulse translated by V t in the periodic box."""
    distance = (x - pulse.x0 - V * t + L / 2) % L - L / 2
    return pulse.amplitude * np.exp(-(distance / pulse.width) ** 2)

spectral = Spectral(L, n_points, D=0.0, V=V)
spectral.initialize(pulse)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4), layout="constrained")
times = [0, 2.5, 5, 9]
for t, color in zip(times, SEQ_CMAP(np.linspace(0.35, 1, len(times)))):
    advance(spectral, t - spectral.t, n_steps=10)
    ax1.plot(spectral.x, spectral.u, color=color, label=f"$t = {t:g}$")
    ax1.plot(spectral.x, advected_pulse(spectral.x, t), "--", color="#0b0b0b", lw=1)
ax1.annotate("", xy=(4.6, 1.08), xytext=(2.6, 1.08), arrowprops=dict(arrowstyle="->", color="#52514e", lw=1.4))
ax1.text(3.6, 1.12, "$V$", ha="center", color="#52514e")
ax1.set(xlabel="$x$", ylabel="$u(x, t)$", xlim=(0, L), ylim=(-0.05, 1.25), title="Pure advection, $D = 0$")
ax1.legend(loc="upper right", fontsize=10, ncols=2)

# advection and diffusion together, recorded at every step
spectral = Spectral(L, n_points, D=0.02, V=V)
spectral.initialize(pulse)
t_grid = np.linspace(0, 25, 501)
u_history = [spectral.u]
for dt in np.diff(t_grid):
    spectral.step(dt)
    u_history.append(spectral.u)
show_spacetime(spectral.x, t_grid, np.array(u_history), ax2, label="$u(x, t)$")
ax2.set_title(f"Advection and diffusion, $D = {spectral.D}$");

spectral = Spectral(L, n_points, D=0.0, V=V)
spectral.initialize(pulse)
advance(spectral, L / V, n_steps=1)
print(f"error after one trip around the box: {np.max(np.abs(spectral.u - pulse(spectral.x))):.1e}")

The pulse travels without any deformation, and when it leaves the box on the right it comes back on the left:
this is the periodic boundary condition built into the Fourier basis. On the space-time map, advection draws a
tilted line of slope $1/V$ that wraps around the box, while diffusion makes it fade and widen. Again, after a full
trip around the box the error is at the level of the round-off.

## Part 2: Finite differences and the Crank-Nicolson method

The spectral method is very accurate, but it has two limitations: it imposes periodic boundary conditions, and it
is exact only for linear equations with constant coefficients. Finite-difference methods work directly on the
grid in real space and do not have these limitations. The Crank-Nicolson (CN) scheme, initially proposed in 1947
for the heat equation, is the classic choice for parabolic equations.

We now solve Eq. (1) with Neumann boundary conditions

\begin{equation}
  \frac{\partial u}{\partial t} + V \frac{\partial u}{\partial x} = D \frac{\partial^2 u}{\partial x^2} + f(u),
  \qquad \left.\frac{\partial u}{\partial x}\right|_{x = 0} = \left.\frac{\partial u}{\partial x}\right|_{x = L} = 0
\end{equation}

A vanishing gradient at the boundaries means that there is no diffusive flux through the walls: for a
concentration, no material can enter or leave the box.

### The grid

We use regular grids in time and space,

\begin{equation}
  t_n = n\, \Delta t, \quad n = 0, \dots, N - 1, \qquad
  x_j = j\, \Delta x, \quad j = 0, \dots, J - 1, \qquad
  \Delta x = \frac{L}{J - 1}
\end{equation}

Contrary to Part 1, both boundaries $x_0 = 0$ and $x_{J-1} = L$ are now grid points. As before, the goal is to
compute numbers $U^n_j \simeq u(x_j, t_n)$.

### The Crank-Nicolson stencil

The time derivative at grid point $(j, n)$ is approximated by a forward difference

\begin{equation}
  \frac{\partial u}{\partial t} \simeq \frac{U^{n+1}_j - U^n_j}{\Delta t}
\end{equation}

If we evaluated the spatial derivatives at time $t_n$ only, we would get an *explicit* scheme: very simple, but
only stable for $D \Delta t / \Delta x^2 \le 1/2$. Halving $\Delta x$ would then force us to divide $\Delta t$ by
four! The idea of Crank and Nicolson is to average the spatial derivatives over the two times $t_n$ and $t_{n+1}$,

\begin{align}
  D \frac{\partial^2 u}{\partial x^2} &\simeq \frac{D}{2 \Delta x^2}
     \Big(U^n_{j+1} - 2 U^n_j + U^n_{j-1} + U^{n+1}_{j+1} - 2 U^{n+1}_j + U^{n+1}_{j-1}\Big) \\
  V \frac{\partial u}{\partial x} &\simeq \frac{V}{4 \Delta x}
     \Big(U^n_{j+1} - U^n_{j-1} + U^{n+1}_{j+1} - U^{n+1}_{j-1}\Big)
\end{align}

so that the scheme is centered at $t_{n+1/2}$. It is then second-order accurate in both $\Delta t$ and $\Delta x$,
and stable for any time step. The six grid points involved form the *stencil* of the method:

![Crank-Nicolson stencil](cn_stencil.png)

The source term is evaluated with the known values, $f(U^n_j)$. Putting everything together:

\begin{equation}
  \frac{U^{n+1}_j - U^n_j}{\Delta t} =
  \frac{D}{2 \Delta x^2} \Big(U^n_{j+1} - 2 U^n_j + U^n_{j-1} + U^{n+1}_{j+1} - 2 U^{n+1}_j + U^{n+1}_{j-1}\Big)
  - \frac{V}{4 \Delta x} \Big(U^n_{j+1} - U^n_{j-1} + U^{n+1}_{j+1} - U^{n+1}_{j-1}\Big) + f(U^n_j)
\end{equation}

### A linear system

With $\sigma \equiv \dfrac{D \Delta t}{2 \Delta x^2}$ and $\nu \equiv \dfrac{V \Delta t}{4 \Delta x}$, we collect
the unknowns $U^{n+1}$ on the left-hand side:

\begin{equation}
  -(\sigma + \nu)\, U^{n+1}_{j-1} + (1 + 2\sigma)\, U^{n+1}_j - (\sigma - \nu)\, U^{n+1}_{j+1}
  = (\sigma + \nu)\, U^n_{j-1} + (1 - 2\sigma)\, U^n_j + (\sigma - \nu)\, U^n_{j+1} + \Delta t\, f(U^n_j)
\end{equation}

This equation holds for $j = 1, \dots, J - 2$. At the boundaries $j = 0$ and $j = J - 1$, it involves the values
$U_{-1}$ and $U_J$ that lie outside the grid. These *ghost points* are fixed by the boundary conditions. With a
centered difference, the Neumann condition reads

\begin{equation}
  \frac{U^n_1 - U^n_{-1}}{2 \Delta x} = 0, \qquad \frac{U^n_J - U^n_{J-2}}{2 \Delta x} = 0
\end{equation}

so we just have to replace $U_{-1} \to U_1$ and $U_J \to U_{J-2}$ (at both times $n$ and $n + 1$): the ghost
point is the mirror image of its neighbor inside the box. In the first and last equations, the coefficients of
$U_1$ and $U_{J-2}$ then become $-2\sigma$ on the left-hand side and $2\sigma$ on the right-hand side. Collecting the values $U^n_j$ in a vector $\mathbf{U}^n$, the
scheme is the linear system

\begin{equation}
  \begin{pmatrix}
    1 + 2\sigma & -2\sigma & & & \\
    -(\sigma + \nu) & 1 + 2\sigma & -(\sigma - \nu) & & \\
    & \ddots & \ddots & \ddots & \\
    & & -(\sigma + \nu) & 1 + 2\sigma & -(\sigma - \nu) \\
    & & & -2\sigma & 1 + 2\sigma
  \end{pmatrix}
  \mathbf{U}^{n+1}
  =
  \begin{pmatrix}
    1 - 2\sigma & 2\sigma & & & \\
    \sigma + \nu & 1 - 2\sigma & \sigma - \nu & & \\
    & \ddots & \ddots & \ddots & \\
    & & \sigma + \nu & 1 - 2\sigma & \sigma - \nu \\
    & & & 2\sigma & 1 - 2\sigma
  \end{pmatrix}
  \mathbf{U}^n
  + \Delta t
  \begin{pmatrix}
    f(U^n_0) \\ f(U^n_1) \\ \vdots \\ f(U^n_{J-2}) \\ f(U^n_{J-1})
  \end{pmatrix}
\end{equation}

that we write as

\begin{equation}
  A\, \mathbf{U}^{n+1} = B\, \mathbf{U}^n + \mathbf{F}^n, \qquad \mathbf{F}^n = \Delta t\, f(\mathbf{U}^n)
  \tag{CN1}
\end{equation}

The matrices $A$ and $B$ only depend on the parameters and on the boundary conditions, so they are built once and
for all. At every time step, we compute the right-hand side, which needs one matrix-vector product, and solve the
linear system for $\mathbf{U}^{n+1}$. A Dirichlet condition such as $u(0, t) = 0$ is even simpler: the first
equation is replaced by $U^{n+1}_0 = 0$, i.e. the first row of $A$ becomes $(1, 0, \dots, 0)$ and the first row of
the right-hand side is set to zero.

How should we solve the linear system? It is tempting to compute $A^{-1}$ once and for all, but this is a bad
idea: $A^{-1}$ is a dense $J \times J$ matrix, and each step would cost $O(J^2)$ operations. The matrix $A$ is
*tridiagonal*, and a tridiagonal system can be solved directly in only $O(J)$ operations. This is what
`scipy.linalg.solve_banded` does.

Finally, without a source term, the CN scheme is second order in time. The explicit evaluation of $f(U^n)$ breaks
this and makes the scheme first order. The second order can be restored by extrapolating the source term to
$t_{n+1/2}$ with the two previous times (this is the Adams-Bashforth method):

\begin{equation}
  A\, \mathbf{U}^{n+1} = B\, \mathbf{U}^n + \frac{3}{2} \mathbf{F}^n - \frac{1}{2} \mathbf{F}^{n-1}
  \tag{CN2}
\end{equation}

### 2.1 A Crank-Nicolson class

Write a class that solves Eq. (CN2). It should

1. create the spatial and temporal grids,
2. store the parameters $D$ and $V$, the function $f$ and the boundary conditions,
3. build the matrices $A$ and $B$ (*hint*: `np.diag(v, k)` creates a matrix with the vector `v` on its `k`-th
   diagonal),
4. modify the first and last rows of $A$ and $B$ to implement Neumann or Dirichlet boundary conditions,
5. solve Eq. (CN2) at every time step with `scipy.linalg.solve`, and return the whole solution as an array
   `u[n, j]` $= U^n_j$.

Here is a possible structure:

```python
class CrankNicolson:
    """Solve du/dt + V du/dx = D d^2u/dx^2 + f(u) with the Crank-Nicolson scheme."""

    def __init__(self, x_min: float, x_max: float, n_x: int, t_max: float, n_t: int) -> None:
        ...

    def set_parameters(self, D: float, V: float, f: Callable[[np.ndarray], np.ndarray] | None = None,
                       boundary_conditions: tuple[str, str] = ("neumann", "neumann")) -> None:
        ...

    def solve(self, u_init: np.ndarray) -> np.ndarray:
        ...
```

Remember that `A @ u` is the matrix-vector product, while `A * u` is an element-wise product.

*Optional*: since $A$ is tridiagonal, you can use `scipy.linalg.solve_banded` instead of `solve`. Read its
documentation carefully: it expects the matrix in a special compact format. How much faster is it?

### Solution

We build $A$ and $B$ as ordinary matrices, which makes them easy to check against the formulas above. For the
fast solver, the function `to_banded` stores the three diagonals of $A$ in the compact format expected by
`solve_banded`, and $B$ is stored as a sparse matrix, so that the product $B \mathbf{U}^n$ only costs $O(J)$
operations. With `method="dense"`, the system is solved with `scipy.linalg.solve`, and with `method="lu"` the
matrix is factorized once with `lu_factor`: this is the right way to "store the inverse" of a matrix. Note that
the matrices are complex as soon as $D$ is complex, we will need this in Part 4.

In [ ]:
def to_banded(M: np.ndarray) -> np.ndarray:
    """Tridiagonal matrix M in the format of solve_banded: ab[1 + i - j, j] = M[i, j]."""
    ab = np.zeros((3, len(M)), dtype=M.dtype)
    ab[0, 1:] = np.diag(M, 1)    # superdiagonal
    ab[1] = np.diag(M)           # diagonal
    ab[2, :-1] = np.diag(M, -1)  # subdiagonal
    return ab


class CrankNicolson:
    """Solve du/dt + V du/dx = D d^2u/dx^2 + f(u) with the Crank-Nicolson scheme."""

    def __init__(self, x_min: float, x_max: float, n_x: int, t_max: float, n_t: int) -> None:
        self.x, self.dx = np.linspace(x_min, x_max, n_x, retstep=True)
        self.t, self.dt = np.linspace(0, t_max, n_t, retstep=True)
        self.set_parameters(D=1.0, V=0.0)

    def set_parameters(self, D: complex, V: float, f: Callable[[np.ndarray], np.ndarray] | None = None,
                       boundary_conditions: tuple[str, str] = ("neumann", "neumann")) -> None:
        self.D, self.V, self.f = D, V, f
        self.boundary_conditions = boundary_conditions

    def matrices(self) -> tuple[np.ndarray, np.ndarray]:
        """The matrices A and B of the scheme, including the boundary conditions."""
        sigma = self.D * self.dt / (2 * self.dx**2)
        nu = self.V * self.dt / (4 * self.dx)
        n = len(self.x)

        def tridiagonal(lower, diagonal, upper):
            return np.diag(np.full(n - 1, lower), -1) + np.diag(np.full(n, diagonal)) + np.diag(np.full(n - 1, upper), 1)

        A = tridiagonal(-(sigma + nu), 1 + 2 * sigma, -(sigma - nu))
        B = tridiagonal(sigma + nu, 1 - 2 * sigma, sigma - nu)

        # boundary rows: (row, neighbor inside the box) at x_min and at x_max
        for (row, neighbor), bc in zip([(0, 1), (n - 1, n - 2)], self.boundary_conditions):
            match bc:
                case "neumann":    # ghost point = mirror image of the neighbor
                    A[row, neighbor], B[row, neighbor] = -2 * sigma, 2 * sigma
                case "dirichlet":  # U_row = 0
                    A[row, :], B[row, :] = 0, 0
                    A[row, row] = 1
                case _:
                    raise ValueError(f"unknown boundary condition {bc!r}")
        return A, B

    def source(self, u: np.ndarray) -> np.ndarray:
        """The source term f(u), set to zero on the Dirichlet boundaries."""
        if self.f is None:
            return np.zeros_like(u)
        f_u = self.f(u) * np.ones_like(u)
        for row, bc in zip([0, -1], self.boundary_conditions):
            if bc == "dirichlet":
                f_u[row] = 0
        return f_u

    def solve(self, u_init: np.ndarray, method: str = "banded", second_order: bool = True) -> np.ndarray:
        """Return the solution u[n, j] = u(x_j, t_n), using (CN2), or (CN1) if second_order is False."""
        A, B = self.matrices()
        match method:
            case "dense":
                solve = lambda rhs: la.solve(A, rhs)
            case "lu":
                lu_and_pivots = la.lu_factor(A)
                solve = lambda rhs: la.lu_solve(lu_and_pivots, rhs)
            case "banded":
                A_banded = to_banded(A)
                B = sparse.csr_array(B)  # only three nonzero elements per row
                solve = lambda rhs: la.solve_banded((1, 1), A_banded, rhs, check_finite=False)
            case _:
                raise ValueError(f"unknown method {method!r}")

        u = np.asarray(u_init, dtype=np.result_type(A, u_init))
        solution = np.empty((len(self.t), len(u)), dtype=u.dtype)
        solution[0] = u
        f_old = self.source(u)
        for n in range(1, len(self.t)):
            f_new = self.source(u)
            f_extrapolated = 1.5 * f_new - 0.5 * f_old if second_order else f_new
            u = solve(B @ u + self.dt * f_extrapolated)
            solution[n] = u
            f_old = f_new
        return solution

Let's look at the matrices on a tiny grid with $\sigma = 1$ and $\nu = 0.25$, to check that the boundary rows are
right:

In [ ]:
tiny = CrankNicolson(x_min=0, x_max=4, n_x=5, t_max=2, n_t=3)  # dx = dt = 1
tiny.set_parameters(D=2.0, V=1.0)                                # sigma = 1, nu = 0.25
A, B = tiny.matrices()
print("A =\n", A, "\n\nB =\n", B)
tiny.set_parameters(D=2.0, V=1.0, boundary_conditions=("dirichlet", "neumann"))
print("\nA with a Dirichlet condition at x_min =\n", tiny.matrices()[0])

### 2.2 Tests

**Initial condition and grids.** Use

1. a Gaussian initial condition with $\sigma = 5$, $x_0 = 50$ and $U_0 = 2$,
2. a spatial grid of 601 points in $[0, 100]$, i.e. $\Delta x = 1/6$,
3. a temporal grid of 1001 points in $[0, 200]$, i.e. $\Delta t = 0.2$.

**Purely diffusive case.** Use $D = 1$, $V = 0$, $f(u) = 0$ and plot the solution $u(x, t)$ as a space-time map
with `show_spacetime`, as well as a few snapshots. At the end of the simulation, the pulse is so wide that it
feels the walls of the box. Can you still find the exact solution? (*Hint*: think of the method of images, a
Neumann wall acts like a mirror.) Is the total amount of tracer $\int u\, dx$ conserved? How long does a
simulation take with `solve`, and with `solve_banded`?

**Purely advective case.** Use $D = 0$, $V = 0.2$, $f(u) = 0$ and compare the final profile with the exact
solution. Then try a narrower pulse, $\sigma = 1$. What do you observe?

### Solution

In [ ]:
x_min, x_max, n_x = 0.0, 100.0, 601
t_max, n_t = 200.0, 1001
crank = CrankNicolson(x_min, x_max, n_x, t_max, n_t)
pulse = Gaussian(width=5.0, amplitude=2.0, x0=50.0)
u_init = pulse(crank.x)
print(f"dx = {crank.dx:.4f}, dt = {crank.dt:.4f}")

#### Diffusion

We first compare the speed of the three ways to solve the linear system:

In [ ]:
# a short run, to time the three solvers
short = CrankNicolson(x_min, x_max, n_x, t_max=50 * crank.dt, n_t=51)
short.set_parameters(D=1.0, V=0.0)
for method in ["dense", "lu", "banded"]:
    start = time.perf_counter()
    short.solve(u_init, method=method)
    print(f"{method:>6}: {1e3 * (time.perf_counter() - start) / 50:7.3f} ms per step")

crank.set_parameters(D=1.0, V=0.0)
u = crank.solve(u_init)

Solving the full system at every step costs $O(J^3)$ operations, and it is by far the slowest. Factorizing $A$ once
makes each step $O(J^2)$, and the banded solver is $O(J)$: here it is more than a hundred times faster than the
dense solver, and the gap grows quickly with $J$.

For the exact solution, a Neumann wall acts like a mirror: the solution in the box is the free solution of the
pulse, plus the free solutions of its mirror images with respect to the walls at $x = 0$ and $x = L$, the images
of the images, and so on. The images are centered at $\pm x_0 + 2mL$ for all integers $m$.

In [ ]:
def diffusing_with_walls(x: np.ndarray, t: float, pulse: Gaussian, D: float, L: float) -> np.ndarray:
    """Exact diffusion of a Gaussian pulse between two Neumann walls at 0 and L (method of images)."""
    images = [DiffusingGaussian(pulse.width, pulse.amplitude, sign * pulse.x0 + 2 * m * L, D)
              for sign in (1, -1) for m in range(-2, 3)]
    return sum(image(x, t) for image in images)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4), layout="constrained")
snapshot_times = [0, 10, 40, 100, 200]
for t, color in zip(snapshot_times, SEQ_CMAP(np.linspace(0.35, 1, len(snapshot_times)))):
    n = np.searchsorted(crank.t, t)
    ax1.plot(crank.x, u[n], color=color, label=f"$t = {t:g}$")
    ax1.plot(crank.x, diffusing_with_walls(crank.x, t, pulse, 1.0, x_max), "--", color="#0b0b0b", lw=1)
free = DiffusingGaussian(pulse.width, pulse.amplitude, pulse.x0, 1.0)
ax1.plot(crank.x, free(crank.x, t_max), ":", color=PALETTE[1], lw=1.6, label="no walls, $t = 200$")
ax1.plot([], [], "--", color="#0b0b0b", lw=1, label="exact, with walls")
ax1.set(xlabel="$x$", ylabel="$u(x, t)$", xlim=(x_min, x_max), title="Diffusion between two walls")
ax1.legend(loc="upper right", fontsize=10)
show_spacetime(crank.x, crank.t, u, ax2, label="$u(x, t)$")
ax2.set_title("Space-time map")

mass = u.sum(axis=1) - 0.5 * (u[:, 0] + u[:, -1])  # trapezoidal rule (times dx)
error = np.max(np.abs(u[-1] - diffusing_with_walls(crank.x, t_max, pulse, 1.0, x_max)))
print(f"max error at t = {t_max:g}: {error:.1e}")
print(f"relative change of the total mass: {np.ptp(mass) / mass[0]:.1e}")

At $t = 200$, the free solution (dotted) has leaked out of the box, while the true solution is reflected by the
walls and piles up near them: it agrees perfectly with the method of images. The total mass is conserved to
machine precision. This is not an accident: summing all the equations of the scheme, with weight $1/2$ for the two
boundary points, all the diffusive fluxes cancel exactly.

#### Advection

With $D = 0$ and $V = 0.2$, the pulse should simply be translated by $V t_\text{max} = 40$. We start it at
$x_0 = 40$, so that it stays away from the wall, and try two pulse widths.

In [ ]:
crank.set_parameters(D=0.0, V=0.2)
widths = [5.0, 1.0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), layout="constrained", sharey=True)
for ax, width in zip(axes, widths):
    narrow = Gaussian(width=width, amplitude=2.0, x0=40.0)
    u = crank.solve(narrow(crank.x))
    ax.plot(crank.x, u[0], color=PALETTE[0], alpha=0.35, label="$t = 0$")
    ax.plot(crank.x, u[-1], color=PALETTE[0], label=f"Crank-Nicolson, $t = {t_max:g}$")
    shifted = Gaussian(width=width, amplitude=2.0, x0=40.0 + crank.V * t_max)
    ax.plot(crank.x, shifted(crank.x), "--", color="#0b0b0b", lw=1.2, label="exact")
    ax.set(xlabel="$x$", xlim=(25, 100), title=fr"$\sigma = {width:g}$  ($\sigma / \Delta x = {width / crank.dx:.0f}$)")
axes[0].set_ylabel("$u(x, t)$")
axes[0].legend(loc="upper left", fontsize=10);

For the wide pulse, the agreement is excellent. But the narrow pulse develops oscillations behind it, while its
height decreases. This is *numerical dispersion*. Consider a plane wave $e^{ikx}$ on the grid: the centered
difference gives $\partial_x e^{ikx} \simeq i \frac{\sin(k\Delta x)}{\Delta x} e^{ikx}$, so the wave travels at the
velocity $V \sin(k \Delta x) / (k \Delta x) < V$. Short waves lag behind! A wide pulse only contains long waves,
with $k \Delta x \ll 1$, and moves correctly. A narrow pulse contains shorter waves that fall behind and form the
wiggles. Including the time discretization, a Fourier analysis of the CN scheme gives the numerical velocity

\begin{equation}
  \frac{v(k)}{V} = \frac{2}{C\, k\Delta x} \arctan\left(\frac{C}{2} \sin(k \Delta x)\right),
  \qquad C = \frac{V \Delta t}{\Delta x}
\end{equation}

where $C$ is the Courant number. Let's plot it together with the spectra of the two pulses:

In [ ]:
courant = crank.V * crank.dt / crank.dx
theta = np.linspace(1e-6, np.pi, 400)  # k dx
velocity = 2 / (courant * theta) * np.arctan(courant / 2 * np.sin(theta))

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(theta, velocity, color="#0b0b0b", label="numerical velocity $v(k) / V$")
for width, color in zip(widths, [PALETTE[0], PALETTE[1]]):
    spectrum = np.exp(-(theta / crank.dx) ** 2 * width**2 / 4)  # Fourier transform of the pulse
    ax.fill_between(theta, spectrum, color=color, alpha=0.3, lw=0, label=fr"spectrum of the pulse, $\sigma = {width:g}$")
ax.set(xlabel=r"$k \Delta x$", ylabel="", xlim=(0, np.pi), ylim=(0, 1.05), title=f"Numerical dispersion ($C = {courant:.2f}$)")
ax.set_xticks([0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi], ["0", r"$\pi/4$", r"$\pi/2$", r"$3\pi/4$", r"$\pi$"])
ax.legend(loc="upper right", fontsize=10);

The wide pulse only contains modes that travel at the right velocity, while a part of the spectrum of the narrow
pulse is significantly slowed down. The CN scheme does not damp any mode (each Fourier mode has an amplification
factor of modulus exactly one for $D = 0$), so the error shows up as oscillations instead of a smearing of the
pulse. The cure is simple: use enough grid points per feature of the solution. Remember that the spectral method of
Part 1 has no such problem, it propagates every mode exactly.

### Going further: how accurate is the scheme?

Let's check the order of convergence in $\Delta t$, with a source term $f(u) = -\gamma u$ that describes a tracer
that slowly decays (or is absorbed by the medium). We compare the solution at $t = 20$ for decreasing time steps
with a reference computed with a much smaller time step, on the same spatial grid, for both (CN1) and (CN2).

In [ ]:
gamma, t_end = 0.05, 20.0
n_steps_list = np.array([10, 20, 40, 80, 160, 320])

def final_profile(n_steps: int, second_order: bool) -> np.ndarray:
    solver = CrankNicolson(x_min, x_max, n_x, t_end, n_steps + 1)
    solver.set_parameters(D=1.0, V=0.0, f=lambda u: -gamma * u)
    return solver.solve(u_init, second_order=second_order)[-1]

reference = final_profile(16 * n_steps_list[-1], second_order=True)
dts = t_end / n_steps_list

fig, ax = plt.subplots()
for second_order, label, color in [(False, "(CN1)", PALETTE[1]), (True, "(CN2)", PALETTE[0])]:
    errors = np.array([np.max(np.abs(final_profile(n, second_order) - reference)) for n in n_steps_list])
    slope, _ = np.polyfit(np.log(dts), np.log(errors), 1)
    ax.loglog(dts, errors, "o-", ms=6, color=color, label=fr"{label}: error $\propto \Delta t^{{{slope:.2f}}}$")
ax.set(xlabel=r"time step $\Delta t$", ylabel="max error at $t = 20$", title="Convergence of the Crank-Nicolson scheme")
ax.legend();

With the explicit source term (CN1), the error decreases like $\Delta t$, while the extrapolated source term (CN2)
recovers the second order: halving the time step divides the error by four. Note also how large the time steps
are: with $\Delta t = 2$ we have $D \Delta t / \Delta x^2 = 72$, way beyond the stability limit $1/2$ of the
explicit scheme, and still the solution is perfectly well behaved.

## Part 3: Nonlinear evolution and invading fronts

We now add the nonlinear source term

\begin{equation}
  f(u) = \mu u - u^3
\end{equation}

For $\mu > 0$, the state $u = 0$ is unstable: a small perturbation grows like $e^{\mu t}$, until the cubic term
saturates it at one of the two stable states $u = \pm\sqrt{\mu}$. Combined with diffusion, a localized
perturbation does not just grow, it spreads and invades the unstable state as two *fronts*. The theory of
such fronts (Fisher, Kolmogorov, Petrovsky and Piskunov, 1937) predicts that they move at the velocity
$v^* = 2\sqrt{D\mu}$ with respect to the medium. With advection, the whole pattern is also carried at the velocity
$V$...

1. Use the parameters $D = 1$, $V = 1.3$, $\mu = 1$ and the boundary conditions $u(0, t) = 0$ (Dirichlet) and
   $\partial_x u(L, t) = 0$ (Neumann). Think of a flow entering the box at $x = 0$.
2. Add the nonlinearity $f(u)$ to your Crank-Nicolson solver, using (CN1) for simplicity or (CN2) to preserve the
   second order.
3. Solve the equation with $u(x, 0)$ a small localized pulse ($U_0 = 0.1$, $\sigma = 1$, $x_0 = 50$), and with a
   small random noise. Plot the space-time maps.
4. Vary the advection velocity $V$. Is there a critical velocity above which the perturbation is washed out of the
   box? Measure the velocity of the fronts and compare with the prediction above. This is the difference between an
   *absolute* and a *convective* instability.

## Solution

Our `source` method sets the source term to zero on a Dirichlet boundary, so that the boundary value stays fixed
whatever the function $f$ (here $f(0) = 0$ anyway). We compare two advection velocities, one below and one above $v^* = 2\sqrt{D\mu} = 2$, for
both a localized seed and a random noise. The dashed lines show the expected fronts,
$x = x_0 + (V \pm v^*)\, t$.

In [ ]:
D, mu = 1.0, 1.0
v_front = 2 * np.sqrt(D * mu)
seed = Gaussian(width=1.0, amplitude=0.1, x0=50.0)
noise = 0.01 * rng.standard_normal(n_x)

fig, axes = plt.subplots(2, 2, figsize=(11, 8), layout="constrained", sharex=True, sharey=True)
for column, V in enumerate([1.3, 3.0]):
    crank.set_parameters(D=D, V=V, f=lambda u: mu * u - u**3, boundary_conditions=("dirichlet", "neumann"))
    for row, (name, u0) in enumerate([("a localized seed", seed(crank.x)), ("a random noise", noise)]):
        ax = axes[row, column]
        show_spacetime(crank.x, crank.t, crank.solve(u0), ax, cmap=DIV_CMAP, vmin=-1, vmax=1, label="$u(x, t)$")
        ax.set_title(f"$V = {V:g}$, starting from {name}", fontsize=13)
    for velocity in (V - v_front, V + v_front):
        axes[0, column].plot(seed.x0 + velocity * crank.t, crank.t, "--", color="#0b0b0b", lw=1.2)
axes[0, 0].set(xlim=(x_min, x_max), ylim=(0, t_max))
for ax in axes[:, 1]:
    ax.set_ylabel("")
for ax in axes[0]:
    ax.set_xlabel("")

For $V = 1.3 < v^*$ (left), the upstream front moves against the flow at the velocity $V - v^* = -0.7$, and the
perturbation eventually invades the whole box: the instability is **absolute**. Starting from noise, the box
quickly fills with domains of $u = +1$ and $u = -1$, the sign being chosen locally by the noise. The kinks between
the domains are carried away by the flow, and the domain that formed at the entrance finally takes over the whole
box.

For $V = 3 > v^*$ (right), both fronts move downstream: the perturbation grows, but it is advected faster than it
spreads, and it is eventually flushed out of the box, leaving the unstable state $u = 0$ behind. The instability is
**convective**: an observer at a fixed position only sees a transient. Starting from noise, the box empties from
the entrance: the noise is amplified while it crosses the box, but cannot survive. The transition between the two
behaviors happens at $V = v^*$.

Let's check this quantitatively by measuring the velocity of the upstream front as a function of $V$. To follow the
front long enough before it hits a boundary, we use a larger box, $x \in [0, 400]$, and we locate the front as the
first point where $u > 1/2$.

In [ ]:
big = CrankNicolson(x_min=0.0, x_max=400.0, n_x=2401, t_max=100.0, n_t=1001)
seed = Gaussian(width=1.0, amplitude=0.1, x0=200.0)
fit_window = (big.t > 30) & (big.t < 90)

velocities = np.linspace(0, 4, 9)
front_velocities = []
for V in velocities:
    big.set_parameters(D=D, V=V, f=lambda u: mu * u - u**3, boundary_conditions=("dirichlet", "neumann"))
    u = big.solve(seed(big.x))
    upstream_front = big.x[np.argmax(u[fit_window] > 0.5, axis=1)]
    front_velocities.append(np.polyfit(big.t[fit_window], upstream_front, 1)[0])

fig, ax = plt.subplots()
ax.axvspan(0, v_front, color="#f0efec", lw=0)
ax.axhline(0, color="#898781", lw=1)
ax.plot(velocities, velocities - v_front, "--", color="#0b0b0b", lw=1.4, label=r"$V - 2\sqrt{D\mu}$")
ax.plot(velocities, front_velocities, "o", ms=7, color=PALETTE[0], label="measured")
ax.text(v_front / 2, 1.9, "absolute", color="#52514e", ha="center")
ax.text(v_front + 1, 1.9, "convective", color="#52514e", ha="center")
ax.set(xlabel="advection velocity $V$", ylabel="velocity of the upstream front", xlim=(-0.2, 4.2), ylim=(-2.3, 2.3), title="Absolute or convective?")
ax.legend(loc="lower right");

The measured velocities follow the prediction $V - 2\sqrt{D\mu}$, and change sign at $V = v^* = 2$. The small
systematic lag is a known property of these *pulled* fronts: they approach their asymptotic velocity very slowly,
as $v(t) \simeq v^* - 3/(2t)$ for $D = \mu = 1$.

The distinction between absolute and convective instabilities is important in fluid mechanics. A wake behind an
obstacle or a jet can amplify a perturbation that is advected away (convective instability, the flow acts as a
noise amplifier) or sustain its own oscillation (absolute instability, the flow is an oscillator).

## Part 4: The complex Ginzburg-Landau equation

\begin{equation}
  \frac{\partial u}{\partial t} + V \frac{\partial u}{\partial x} = (1 + i\alpha) \frac{\partial^2 u}{\partial x^2}
  + \mu u - (1 + i\beta)\, |u|^2 u
  \tag{2}
\end{equation}

This is the complex Ginzburg-Landau equation (CGLE). It is one of the most studied equations in nonlinear physics,
because it describes *any* spatially extended system close to an oscillatory instability (a Hopf bifurcation):
convection in binary fluid mixtures, oscillating chemical reactions such as the Belousov-Zhabotinsky reaction,
lasers, and many more. The field $u$ is complex: $|u|$ is the amplitude of the oscillation and $\arg u$ its phase.
For $\alpha = \beta = 0$, it reduces to the real Ginzburg-Landau equation, familiar from superconductivity.

For $\mu > 0$, the CGLE has plane-wave solutions $u = \sqrt{\mu - q^2}\, e^{i(qx - \omega t)}$. The uniform
oscillation $q = 0$ is linearly stable when $1 + \alpha\beta > 0$, and unstable otherwise: this is the
*Benjamin-Feir-Newell* criterion. Beyond this line, there are no stable plane waves and the system becomes
spatiotemporally chaotic. Depending on $\alpha$ and $\beta$, different kinds of chaos are observed, as summarized in
this phase diagram (adapted from H. Chaté, Nonlinearity **7**, 185 (1994); the line marked BF is the
Benjamin-Feir-Newell line):

![Phase diagram of the complex Ginzburg-Landau equation](phasediagram.jpg)

- In **phase turbulence**, the amplitude $|u|$ stays away from zero, and only the phase fluctuates, smoothly.
- In **defect turbulence**, the amplitude regularly vanishes at isolated points of space-time. There, the phase is
  undefined and it can jump by $2\pi$: these events are called *phase slips* or *defects*.
- In **spatiotemporal intermittency**, turbulent patches coexist with laminar regions, and they keep appearing and
  disappearing.

A nice review of the many faces of this equation is I. S. Aranson and L. Kramer, Rev. Mod. Phys. **74**, 99 (2002).

### 4.1 Complex Crank-Nicolson

Here we go back to the Neumann boundary conditions $\partial_x u = 0$ at $x = 0$ and $x = L$, without advection
($V = 0$), and with $\mu = 1$.

1. Modify the construction of the matrices $A$ and $B$ (if necessary) so that they can hold complex numbers.
   Equation (2) is Eq. (1) with a complex diffusion coefficient $D = 1 + i\alpha$.
2. Modify the function $f(u)$ to match Eq. (2).
3. Check your code with $\alpha = 0$, $\beta = -1.77$. As initial condition, use the sum of two Gaussians with
   $\sigma = 1$, centered at $x_0 = 60$ and $x_1 = 30$, with amplitudes $U_0 = 0.1 + 0.2i$ and $U_0/2$. Plot the
   real part $\mathrm{Re}\, u(x, t)$, the amplitude $|u(x, t)|$ and the phase $\arg u(x, t)$.

*Hints*: a cyclic colormap such as `"twilight"` is best for a phase. Once the oscillation has saturated, the whole
system oscillates in time at the frequency $\beta$ (check that $u = e^{-i\beta t}$ solves Eq. (2) for $\mu = 1$).
To see the spatial structure of the phase, it is more instructive to plot the phase of $u\, e^{i\beta t}$.

### Solution

Our `CrankNicolson` class already works with complex numbers: the matrices take the type of $\sigma$, which is
complex when $D$ is complex, and the solution array takes the common type of the matrices and of the initial
condition. We only need a new source term. Since we will make several such plots, we write a function that draws
the real part, the amplitude and the phase of $u$ side by side.

In [ ]:
def ginzburg_landau(alpha: float, beta: float, mu: float = 1.0) -> Callable[[np.ndarray], np.ndarray]:
    """The nonlinear term of the complex Ginzburg-Landau equation."""
    return lambda u: mu * u - (1 + 1j * beta) * np.abs(u) ** 2 * u


def plot_cgle(x: np.ndarray, t: np.ndarray, u: np.ndarray, title: str, omega: float = 0.0) -> np.ndarray:
    """Space-time maps of the real part, the amplitude and the phase of u (in the frame rotating at omega)."""
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.6), layout="constrained", sharey=True)
    re_max = np.abs(u.real).max()
    show_spacetime(x, t, u.real, axes[0], cmap=DIV_CMAP, vmin=-re_max, vmax=re_max, label=r"$\mathrm{Re}\, u$")
    show_spacetime(x, t, np.abs(u), axes[1], vmin=0, label="$|u|$")
    phase = np.angle(u * np.exp(1j * omega * t)[:, None])
    image = show_spacetime(x, t, phase, axes[2], cmap=PHASE_CMAP, vmin=-np.pi, vmax=np.pi, label=r"$\arg u$")
    image.colorbar.set_ticks([-np.pi, 0, np.pi], labels=[r"$-\pi$", "0", r"$\pi$"])
    for ax, name in zip(axes, ["real part", "amplitude", "phase"]):
        ax.set_title(name)
    for ax in axes[1:]:
        ax.set_ylabel("")
    fig.suptitle(title, fontsize=15)
    return axes


alpha, beta = 0.0, -1.77
crank.set_parameters(D=1 + 1j * alpha, V=0.0, f=ginzburg_landau(alpha, beta))
U0 = 0.1 + 0.2j
two_pulses = U0 * (np.exp(-(crank.x - 60) ** 2) + 0.5 * np.exp(-(crank.x - 30) ** 2))
u = crank.solve(two_pulses)
plot_cgle(crank.x, crank.t, u, fr"$\alpha = {alpha:g}$, $\beta = {beta:g}$, starting from two pulses", omega=beta);


The two small pulses first grow into oscillations, which spread as fronts and fill the box by $t \simeq 25$.
After that, the plane waves are linearly stable here ($1 + \alpha\beta = 1 > 0$), and yet the system never
settles. Look at the amplitude: dark regions where $|u| \simeq 1$ (laminar, a regular oscillation) coexist with
bright localized dips of $|u|$ that wander, branch and die. On the phase map, the laminar regions are made of
plane waves, which appear as tilted stripes, and the dips are where the phase changes abruptly. This
mixture of laminar and turbulent regions is called **spatiotemporal intermittency**, and it is indeed the region of
the phase diagram where $\alpha = 0$, $\beta = -1.77$ lies. Since the plane waves are linearly stable, this
chaos needs a finite perturbation to start: it is a nonlinear instability.


### 4.2 Spatiotemporal chaos

1. Simulate the regime of **defect turbulence**, $\alpha = 2$, $\beta = -2$, starting from a small complex random
   noise. You can use a larger box, e.g. $L = 200$, and a longer time, to see the chaos develop.
2. Simulate the regime of **phase turbulence**, $\alpha = 2$, $\beta = -0.8$, again starting from a small random
   noise. What changes if you take $\beta = -1$?
3. Where are these two points in the phase diagram above? How can you tell the two regimes apart on your plots?
   Make a histogram of $|u|$ in both cases, once the initial transient is over.
4. *Optional*: make an animation of $\mathrm{Re}\, u$, $\mathrm{Im}\, u$ and $|u|$ as a function of $x$, using
   `matplotlib.animation.FuncAnimation` as in Project 2.

### Solution

We use a box twice as large as before, $L = 200$, and integrate up to $t = 500$. The initial condition is a small
complex noise. In both regimes, the uniform state $u = 0$ is unstable and the noise first grows into an oscillation
of amplitude $|u| \simeq 1$; what happens next depends on $\alpha$ and $\beta$.

In [ ]:
cgle = CrankNicolson(x_min=0.0, x_max=200.0, n_x=1001, t_max=500.0, n_t=2501)
noise = 0.01 * (rng.standard_normal(len(cgle.x)) + 1j * rng.standard_normal(len(cgle.x)))

regimes = {"defect turbulence": (2.0, -2.0), "phase turbulence": (2.0, -0.8)}
solutions = {}
for name, (alpha, beta) in regimes.items():
    cgle.set_parameters(D=1 + 1j * alpha, V=0.0, f=ginzburg_landau(alpha, beta))
    solutions[name] = cgle.solve(noise)
    plot_cgle(cgle.x, cgle.t, solutions[name], fr"{name.capitalize()}: $\alpha = {alpha:g}$, $\beta = {beta:g}$", omega=beta)

The two regimes look very different. In **defect turbulence** (first figure), the amplitude is disordered on
short scales in space and time, and it often drops to zero: the phase is then completely scrambled. In **phase
turbulence** (second figure), the amplitude remains close to one everywhere, and only shallow dips wander
through the system. The phase varies smoothly, with large-scale fluctuations that never die out.

Note that the real part mostly shows the fast oscillation of the whole system at the frequency $\beta$: the uniform
solution of Eq. (2) is $u = e^{-i\beta t}$ for $\mu = 1$. To see the interesting dynamics of the phase, we have
plotted it in the frame that rotates with this uniform oscillation, i.e. the phase of $u\, e^{i\beta t}$ (this is
the `omega` argument of `plot_cgle`).

The histogram of $|u|$, once the initial transient is over, is the clearest way to tell the two regimes apart:

In [ ]:
fig, ax = plt.subplots()
bins = np.linspace(0, 1.6, 161)
late = cgle.t > 100
for (name, u), color in zip(solutions.items(), [PALETTE[1], PALETTE[0]]):
    ax.hist(np.abs(u[late]).ravel(), bins=bins, density=True, histtype="stepfilled", color=color, alpha=0.7, label=name)
    print(f"{name:>17}: min |u| = {np.abs(u[late]).min():.3f}")
ax.set(xlabel="amplitude $|u|$", ylabel="probability density", yscale="log", title="Distribution of the amplitude")
ax.legend(loc="upper left");

In phase turbulence, $|u|$ never goes below $\sim 0.9$: the phase is always well defined and it can never jump.
In defect turbulence, the distribution extends all the way down to $|u| = 0$, where the defects are born. The
parameters $\alpha = 2$, $\beta = -0.8$ lie in the narrow band of phase turbulence along the Benjamin-Feir-Newell
line of the phase diagram ($1 + \alpha\beta = -0.6 < 0$). You can try $\beta = -1$, slightly further from the line:
there, defects occasionally appear and the system becomes a mixture of the two regimes.

Finally, here is an animation of the defect turbulence. Look for the moments when $|u|$ touches zero: at that
point, the real and imaginary parts vanish together and the phase slips by $2\pi$.

In [ ]:
def animate_cgle(x: np.ndarray, t: np.ndarray, u: np.ndarray, frames: range) -> animation.FuncAnimation:
    """Animate the real part, imaginary part and amplitude of u(x, t)."""
    fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained", dpi=80)
    (line_re,) = ax.plot(x, u[frames[0]].real, color=PALETTE[0], lw=1.4, label=r"$\mathrm{Re}\, u$")
    (line_im,) = ax.plot(x, u[frames[0]].imag, color=PALETTE[1], lw=1.4, label=r"$\mathrm{Im}\, u$")
    (line_abs,) = ax.plot(x, np.abs(u[frames[0]]), color="#0b0b0b", lw=2.2, label="$|u|$")
    title = ax.set_title("")
    ax.axhline(0, color="#898781", lw=1)
    ax.set(xlim=(x[0], x[-1]), ylim=(-1.6, 1.6), xlabel="$x$")
    ax.legend(loc="upper right", ncols=3, frameon=True, facecolor="white", edgecolor="none")

    def update(n: int):
        line_re.set_ydata(u[n].real)
        line_im.set_ydata(u[n].imag)
        line_abs.set_ydata(np.abs(u[n]))
        title.set_text(f"Defect turbulence, $t = {t[n]:.0f}$")
        return line_re, line_im, line_abs, title

    ani = animation.FuncAnimation(fig, update, frames=frames, interval=60)
    plt.close(fig)
    return ani


# rendering the animation takes about 30 seconds
HTML(animate_cgle(cgle.x, cgle.t, solutions["defect turbulence"], frames=range(1000, 1300, 2)).to_jshtml())

You can compare these results with the figures of the review by Aranson and Kramer. And there is much more to
explore: try other points of the phase diagram (for instance spatiotemporal intermittency at $\alpha = 0$,
$\beta = -3$), switch on the advection $V$ to see whether the chaos is absolute or convective as in Part 3, or
start from the uniform oscillation $u = 1$ slightly perturbed by noise, and watch it destabilize when
$1 + \alpha\beta < 0$.